In [0]:
from pyspark.sql.functions import col, sum, count, round, avg

orders = spark.table("silver.olist.orders")
order_items = spark.table("silver.olist.order_items")
sellers = spark.table("silver.olist.sellers")
reviews = spark.table("silver.olist.order_reviews")

# Join items + orders + sellers
df = order_items \
    .join(orders.select("order_id", "order_status"), on="order_id", how="left") \
    .join(sellers, on="seller_id", how="left")

# Join reviews
df = df.join(
    reviews.select("order_id", "review_score"),
    on="order_id", how="left"
)

# Aggregate per seller
gold_sellers = df.groupBy("seller_id", "seller_city", "seller_state") \
    .agg(
        round(sum("price"), 2).alias("total_revenue"),
        count("order_id").alias("total_orders"),
        round(avg("review_score"), 2).alias("avg_review_score"),
        round(avg("freight_value"), 2).alias("avg_freight_charged")
    ) \
    .orderBy(col("total_revenue").desc())

# Write back to Gold layer
gold_sellers.write.format("delta").mode("overwrite") \
    .option("path", "abfss://gold@project1storagek.dfs.core.windows.net/seller_performance") \
    .saveAsTable("gold.olist.seller_performance")

print("seller_performance done:", gold_sellers.count())
gold_sellers.show(10)

seller_performance done: 3095
+--------------------+----------------+------------+-------------+------------+----------------+-------------------+
|           seller_id|     seller_city|seller_state|total_revenue|total_orders|avg_review_score|avg_freight_charged|
+--------------------+----------------+------------+-------------+------------+----------------+-------------------+
|4869f7a5dfa277a7d...|         guariba|          SP|    229472.63|        1156|            4.12|              17.45|
|53243585a1d6dc264...|lauro de freitas|          BA|    222776.05|         410|            4.08|               31.9|
|4a3ca9315b744ce9f...|        ibitinga|          SP|    202222.32|        2001|            3.81|              17.66|
|fa1c13f2614d7b5c4...|          sumare|          SP|    194042.03|         586|            4.34|              17.14|
|7c67e1448b00f6e96...| itaquaquecetuba|          SP|    188720.76|        1371|            3.35|              37.78|
|7e93a43ef30c4f03f...|         bar

In [0]:
spark.sql("DROP TABLE IF EXISTS gold.olist.seller_performance")

DataFrame[]